# 21 · Il device si collega a Event Grid con il certificato di ADR

Il device apre una connessione **MQTT su TLS** (porta 8883) presentando il suo certificato, e manda il primo pacchetto, `CONNECT`. Il broker risponde `CONNACK`: **0 vuol dire accettato**, **5 non autorizzato**.
Non pubblichiamo né sottoscriviamo niente: per sapere se un certificato vale basta il collegamento.

Quattro prove:
1. **eventgrid-sensore-01 con il certificato di ADR, censito per impronta**.
2. **lo stesso certificato, con il client censito per CA** (la CA di ADR è caricata, il nome è quello del certificato).
3. **il certificato di bootstrap**: è firmato da un'altra CA, e il suo thumbprint non è quello censito.
4. **eventgrid-non-censito-01 con il certificato di ADR, ma non censito**: il certificato è valido, il client non esiste.

**Serve:** il notebook 20 già eseguito (crea il namespace, registra i due device, censisce i client).
**Attenzione:** Event Grid ci mette qualche decina di secondi a usare un client appena creato. Se la prima prova risponde 5 subito dopo il notebook 20, aspetta mezzo minuto e riprova.

**Da dire.** "Lo stesso certificato che apre l'Hub apre anche questo broker, ma dipende da come lo censisco. Con l'impronta entra. Con la CA no: lo stesso controllo, fatto da Event Grid, dice di no."


In [ ]:
# Carica la libreria e legge lo scenario (notebooks\_scenario.psd1): da qui in poi si usa $cfg
. ([scriptblock]::Create([IO.File]::ReadAllText((Get-ChildItem . , .\notebooks , .. -Filter _start.ps1 -ErrorAction SilentlyContinue | Select-Object -First 1).FullName)))

In [ ]:
if (-not $cfg -or -not $out) { throw "Mancano le variabili dello scenario: esegui prima la cella di caricamento (la prima cella di codice) di questo notebook." }
$eg = $cfg.EventGridName
$ns = az eventgrid namespace show --name $eg --resource-group $rg -o json --only-show-errors | ConvertFrom-Json
Assert-Az "Lettura del namespace $eg (esegui prima il notebook 20)"
$mqttHost = $ns.topicSpacesConfiguration.hostname
if (-not $mqttHost) { throw "Non trovo l'indirizzo MQTT di ${eg}: esegui prima il notebook 20." }
function Cartella($id) { (Get-ChildItem (Join-Path $out "devices\$id") -Directory -ErrorAction SilentlyContinue | Sort-Object Name | Select-Object -Last 1).FullName }
$sensore = Cartella 'eventgrid-sensore-01'
$nonCensito = Cartella 'eventgrid-non-censito-01'
$sensoreImpronta = 'eventgrid-sensore-01-impronta'   # il client censito per impronta (notebook 20)
"Broker: $mqttHost | eventgrid-sensore-01: $sensore"

## 1 · Per impronta: eventgrid-sensore-01 con il certificato di ADR
Il client `eventgrid-sensore-01-impronta` conosce l'impronta di questo certificato. Deve entrare: risposta `accepted`.


In [ ]:
'{0,-62} {1}' -f 'certificato di ADR, client per impronta', (Test-MqttBrokerAcceptsCertificate -Hostname $mqttHost -DeviceId $sensoreImpronta -StateDir $sensore)

## 2 · Per CA: lo stesso certificato
Il client `eventgrid-sensore-01` si fida della **CA di ADR** e il suo nome è uguale al nome del certificato (`CN=eventgrid-sensore-01`). Sulla carta il certificato dovrebbe entrare.


In [ ]:
'{0,-62} {1}' -f 'certificato di ADR, client per CA', (Test-MqttBrokerAcceptsCertificate -Hostname $mqttHost -DeviceId 'eventgrid-sensore-01' -StateDir $sensore)

## 3 · Controprova: il certificato di bootstrap
Stesso device, ma il certificato di **bootstrap**: la sua CA è quella che hai creato tu per DPS, non quella di ADR. Event Grid non lo conosce.
Non risponde con un errore di TLS: lascia passare la connessione TLS e risponde `code 5` al `CONNECT`.


In [ ]:
'{0,-62} {1}' -f 'certificato di bootstrap (altra CA)', (Test-MqttBrokerAcceptsCertificate -Hostname $mqttHost -DeviceId $sensoreImpronta -StateDir $sensore -CertFile 'bootstrap-cert.pem' -KeyFile 'bootstrap-key.pem')

## 4 · Controprova: un device non censito
`eventgrid-non-censito-01` ha un certificato di ADR valido, ma non è un client di Event Grid. Se il suo certificato non era stato ancora emesso la cella lo dice.

In [ ]:
if (-not $nonCensito) { "eventgrid-non-censito-01 non ha un certificato: esegui prima il notebook 20." }
else { '{0,-62} {1}' -f 'eventgrid-non-censito-01, certificato di ADR, NON censito', (Test-MqttBrokerAcceptsCertificate -Hostname $mqttHost -DeviceId 'eventgrid-non-censito-01' -StateDir $nonCensito) }

## Cosa abbiamo visto
- Il certificato di **ADR vale anche fuori dall'Hub**: Event Grid lo accetta se il client è censito **per impronta**.
- **Per CA no.** Anche con la CA di ADR caricata e il nome giusto, Event Grid risponde `code 5`. Un controllo fatto fuori da questo notebook (2026-11-22) indica la causa: una catena nostra, con le stesse caratteristiche di quella di ADR, viene accettata; la stessa catena con in più la sola **estensione critica di Microsoft** (`1.3.6.1.4.1.311.133.1.1.1`) viene rifiutata. È la stessa estensione che fa rifiutare il certificato a .NET (notebook 22).
- **Event Grid non spiega perché**: un'altra CA, un client sconosciuto o un'estensione che non accetta danno tutti `code 5`. E non rifiuta nella stretta di mano TLS.
- Il prezzo dell'impronta: ogni rinnovo cambia il certificato (30 giorni qui) e quindi l'impronta, che va aggiornata nel client.

**Non provato:**
- cosa succede se il certificato viene **revocato** in ADR: un client per impronta non guarda nessuna lista di revoca, quindi lo accetterebbe ancora? Non l'ho provato. (La revoca vista da un servizio che la controlla: notebook 22.)
- cosa succede se il device viene **disabilitato** in ADR: Event Grid non lo sa, perché non è collegato ad ADR.
- un altro modo di censire, per esempio con i nomi alternativi nel certificato (SAN): non provato.

**Prossimo notebook: 22, un'API tua che si fida degli stessi certificati.**
